In [7]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import wandb
import numpy as np
import os

In [2]:
SITE_INFO = [
    "site_id", "observation_hour", "station_name"
]

STATIC_FEATURES = [
    "longitude", "latitude", "DRAIN_SQKM", "artificial_path_pct",
    "wb5100_ann_mm", "snw_pc_syr", "snow_ice_nlcd06", "barren_nlcd06",
    "mains100_plant", "hga", "hgc", "bulk_density_avg", "elev_max_m", "aspect_deg"
]

DYNAMIC_FEATURES = [
    "streamflow_cfs_mean", "streamflow_cfs_max", "streamflow_cfs_min",
    "gage_height_ft_mean",
    "precipitation_mm",
    "temperature_c",
    "potential_evaporation_mm",
    "specific_humidity_kgkg",
    "shortwave_radiation_wm2",
    "longwave_radiation_wm2",
    "wind_speed_ms",
    "surface_pressure_pa",
    "cape_jkg",
    "convective_precip_fraction",
]

TARGET = "streamflow_cfs_mean" 

In [17]:
lp3_df = pl.read_csv('../../elt/transformation/seeds/lp3_results.csv')
lp3_sites_df = lp3_df['site_id'].to_list()
lp3_sites_df = [str(s) for s in lp3_sites_df]
len(lp3_sites_df)

559

In [16]:
api = wandb.Api()
artifact = api.artifact("flood-forecasting/flood-dataset:latest")
artifact_dir = artifact.download()

LOAD_COLS = SITE_INFO + STATIC_FEATURES + DYNAMIC_FEATURES

df = (
    pl.scan_parquet(f"{artifact_dir}/flood_model.parquet")
    .select(LOAD_COLS)
    .filter(pl.col("site_id").is_in(lp3_sites_df))
    .collect()
)

print(f"Full dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total sites: {df['site_id'].n_unique()}")
print(f"Date range: {df['observation_hour'].min()} to {df['observation_hour'].max()}")

wandb: Downloading large artifact 'flood-dataset:latest', 4549.24MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (24198.1MB/s)


Full dataset: 372,421 rows x 31 columns
Total sites: 3
Date range: 2007-10-28 05:00:00 to 2026-02-07 05:00:00


In [ ]:
# Distribution of null rates
null_data = site_stats.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Streamflow Null %", "Gage Height Null %"))

fig.add_trace(
    go.Histogram(x=null_data["streamflow_null_pct"], nbinsx=50, name="Streamflow", marker_color="steelblue"),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=null_data["gage_height_null_pct"], nbinsx=50, name="Gage Height", marker_color="darkorange"),
    row=1, col=2
)
fig.update_layout(
    title="Distribution of Null Rates Across Sites",
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
)
fig.update_xaxes(title_text="Percents (%)", showline=True, linecolor="black", linewidth=1, mirror=True)
fig.update_yaxes(title_text="Number of Sites",
                 showline=True, linecolor="black", linewidth=1, mirror=True,
                 showgrid=True, gridcolor="lightgrey", gridwidth=0.5)
fig.show()

# Summary
print(f"Sites with 0% streamflow nulls:    {len(null_data[null_data['streamflow_null_pct'] == 0])}")
print(f"Sites with <20% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] < 20])}")
print(f"Sites with >50% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] > 50])}")
print(f"Sites with 100% streamflow nulls:  {len(null_data[null_data['streamflow_null_pct'] == 100])}")
print()
print(f"Sites with 0% gage height nulls:   {len(null_data[null_data['gage_height_null_pct'] == 0])}")
print(f"Sites with <20% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] < 20])}")
print(f"Sites with >50% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] > 50])}")
print(f"Sites with 100% gage height nulls: {len(null_data[null_data['gage_height_null_pct'] == 100])}")

fig.write_html("sacha_null_histogram.html")